# Cómo interpretar y convertir tipos de datos

En esta práctica trabajaremos con trabajar con otro aspecto fundamental de la preparación de datos: los tipos de datos.

Cuando Pandas carga un archivo, intenta interpretar automáticamente qué tipo de información contiene cada columna. Algunas columnas quedan como números, otras como texto y otras pueden necesitar una conversión especial. Aun así, esa interpretación automática no siempre coincide con lo que es necesario para analizar.

Una columna puede “parecer” numérica porque contiene valores como `2`, `3.5` o `1200`, pero si está cargada como texto, Pandas no la tratará como una variable numérica real. Algo similar ocurre con las fechas: una fecha puede verse como `2024-03-01`, pero si Pandas la interpreta como texto, no podremos trabajarla correctamente como dato temporal.

En este capítulo trabajaremos con revisar tipos de datos, detectar columnas que necesitan conversión y transformar valores usando herramientas como `pd.to_numeric()` y `pd.to_datetime()`.

Al finalizar este notebook deberías poder:

- Comprender por qué los tipos de datos son relevantes para el análisis.
- Revisar los tipos de columnas con `dtypes` e `info()`.
- Detectar columnas numéricas cargadas como texto.
- Convertir columnas a formato numérico usando `pd.to_numeric()`.
- Comprender el uso de `errors="coerce"`.
- Verificar si la conversión generó nuevos valores faltantes.
- Convertir una columna de texto a fecha usando `pd.to_datetime()`.
- Validar la estructura del `DataFrame` después de convertir tipos.

### Punto de control

El tipo de una columna determina qué operaciones puede realizar Pandas con seguridad. Por eso la conversión debe evaluarse como parte del diagnóstico y no como un cambio meramente visual.

## Fuente para la práctica

Para estudiar conversión de tipos de datos trabajaremos con volver al dataset real **Cafe Sales — Dirty Data for Cleaning Training**.

Este dataset contiene transacciones de venta de una cafetería. Algunas columnas deberían poder analizarse como valores numéricos, por ejemplo `Quantity`, `Price Per Unit` y `Total Spent`. También hay una columna de fecha, `Transaction Date`, que debería poder interpretarse como dato temporal.

Aun así, como el dataset contiene valores problemáticos, Pandas puede cargar algunas de estas columnas con tipos de datos que no son los más adecuados para el análisis.

Como cada capítulo debe poder ejecutarse de manera independiente, trabajaremos con descargar y cargar nuevamente el dataset.

### Punto de control

Observar algunos registros es útil, pero no basta para confirmar un formato. Los valores excepcionales suelen aparecer fuera de la muestra y pueden ser precisamente los que impiden una conversión completa.

In [ ]:
# Carga manual del archivo en Google Colab

from google.colab import files
import io
import pandas as pd

# El usuario selecciona desde su equipo el CSV que desea analizar.
uploaded = files.upload()
archivos = list(uploaded.keys())
archivos_csv = [archivo for archivo in archivos if archivo.lower().endswith(".csv")]

if not archivos_csv:
    raise ValueError("Debes subir al menos un archivo con extensión .csv")

nombre_csv = archivos_csv[0]
df = pd.read_csv(io.BytesIO(uploaded[nombre_csv]))

df.head()

La salida de `head()` nos permite confirmar que el archivo fue cargado correctamente.

En este capítulo no trabajaremos con concentrarnos todavía en limpiar todos los problemas del dataset. El foco estará puesto en revisar cómo Pandas interpretó los tipos de datos y en convertir algunas columnas para que puedan usarse correctamente en cálculos y análisis posteriores.

### Punto de control

Una conversión controlada conserva la evidencia de los casos que no pudieron interpretarse. Es preferible conocer esos casos y revisarlos que ocultarlos mediante una transformación silenciosa.

## Diagnóstico de tipos

Después de cargar un dataset, una de las primeras tareas de diagnóstico consiste en revisar qué tipo de dato asignó Pandas a cada columna.

Esto es relevante porque muchas operaciones dependen del tipo de dato. Si una columna fue interpretada como numérica, podremos calcular promedios, medianas, sumas o rankings. Si una columna fue interpretada como texto, esas operaciones pueden fallar o producir resultados que no tienen sentido.

Podemos revisar los tipos de datos con `dtypes`.

### Punto de control

Las copias permiten comparar el estado original con la versión convertida. Esta práctica facilita regresar a la fuente y probar otra estrategia si el resultado no coincide con el objetivo del análisis.

In [ ]:
df.dtypes

El resultado permite observar el tipo de dato de cada columna.

En Pandas, el tipo `object` suele indicar texto o valores mezclados. No siempre significa que la columna esté mal, pero sí nos invita a revisar con cuidado.

En este dataset, columnas como `Item`, `Payment Method` o `Location` pueden ser textuales. Eso es esperable. Pero si columnas como `Quantity`, `Price Per Unit` o `Total Spent` aparecen como `object`, tenemos una señal de problema: son columnas que deberían poder analizarse como numéricas.

También es posible usar `info()` para obtener una visión general de tipos y valores no nulos.

### Punto de control

Después de transformar una columna, la validación debe revisar tanto el tipo obtenido como los valores faltantes nuevos. Un tipo correcto no garantiza que todos los registros hayan sido interpretados correctamente.

In [ ]:
df.info()

La salida de `info()` permite ver, en una misma tabla, el nombre de cada columna, la cantidad de valores no nulos y el tipo de dato asignado por Pandas.

Esta revisión es parte del diagnóstico inicial. Antes de convertir columnas, es necesario identificar cuáles parecen tener un tipo de dato inadecuado y por qué eso puede afectar el análisis.

### Punto de control

Las fechas requieren atención especial porque una cadena puede parecer ordenada y, aun así, no comportarse como una variable temporal. Convertirlas habilita filtros, agrupaciones y comparaciones cronológicas confiables.

## Leer la estructura detectada

La salida de `info()` nos muestra algo relevante: todas las columnas del dataset fueron cargadas con tipo `object`.

Esto significa que Pandas está tratando todas las columnas como texto o como columnas con valores mezclados.

En algunas columnas esto es esperable. Por ejemplo, `Transaction ID` es un identificador, por lo tanto no es necesario tratarlo como número. Lo mismo ocurre con `Item`, `Payment Method` y `Location`, que representan categorías o textos.

Pero hay otras columnas que deberían poder analizarse de otra manera:

```text
Quantity           → debería ser numérica
Price Per Unit     → debería ser numérica
Total Spent        → debería ser numérica
Transaction Date   → debería ser una fecha
```

Si `Quantity`, `Price Per Unit` y `Total Spent` permanecen como `object`, no podremos trabajar con ellas de manera confiable como columnas numéricas. Esto puede afectar cálculos como sumas, promedios, medianas, rankings o validaciones entre columnas.

Algo similar ocurre con `Transaction Date`. Aunque sus valores parezcan fechas, mientras Pandas la interprete como `object`, esa columna sigue siendo texto. Para analizarla temporalmente, ordenar fechas, extraer meses o agrupar ventas por día, trabajaremos con necesitar convertirla a un tipo de dato temporal.

También vemos que varias columnas tienen menos de 10000 valores no nulos. Eso indica que hay valores faltantes. En este capítulo no trabajaremos con centrarnos en el tratamiento de faltantes, porque ya lo trabajamos antes. Aun así, al convertir tipos debemos tenerlos en cuenta, porque algunos errores de conversión también pueden generar nuevos valores faltantes.

Antes de analizar este dataset, es necesario convertir algunas columnas al tipo de dato adecuado.

### Punto de control

La calidad del resultado depende de los supuestos utilizados durante la conversión. Cuando aparecen `ERROR`, `UNKNOWN` u otros textos, es necesario distinguirlos de los datos válidos antes de continuar.

## Identificar variables numéricas

Antes de convertir columnas, resulta útil observar algunos valores.

En este dataset hay tres columnas que deberían representar cantidades o importes:

```text
Quantity
Price Per Unit
Total Spent
```

Aunque visualmente puedan parecer numéricas, Pandas las cargó como `object`. Eso suele ocurrir cuando una columna contiene números mezclados con textos, valores problemáticos o datos faltantes.

Vamos a observar algunos valores de esas columnas.

### Punto de control

El objetivo no es forzar todas las columnas a un tipo único. Cada variable debe conservar el tipo que corresponda a su significado y a las operaciones que se realizarán después.

In [ ]:
df[["Quantity", "Price Per Unit", "Total Spent"]].head(15)

Esta vista nos permite revisar ejemplos concretos.

Si una columna contiene valores como `"ERROR"` o `"UNKNOWN"`, Pandas no puede convertirla automáticamente a número durante la carga del archivo. Por esa razón la conserva como `object`.

También es posible revisar las frecuencias de valores en cada una de estas columnas. Esto puede ayudarnos a detectar valores problemáticos.

### Punto de control

El tipo de una columna determina qué operaciones puede realizar Pandas con seguridad. Por eso la conversión debe evaluarse como parte del diagnóstico y no como un cambio meramente visual.

In [ ]:
df["Quantity"].value_counts(dropna=False).head(15)

In [ ]:
df["Price Per Unit"].value_counts(dropna=False).head(15)

In [ ]:
df["Total Spent"].value_counts(dropna=False).head(15)

En estas salidas debemos prestar atención a valores que no sean números reales.

Si aparecen valores como `"UNKNOWN"` o `"ERROR"`, ya tenemos una explicación para el tipo `object`: la columna mezcla valores numéricos con textos problemáticos.

Antes de convertir, es relevante detectar estos casos. Convertir tipos no debería ser una acción ciega. Necesitamos saber qué clase de valores contiene la columna y qué ocurrirá con aquellos que no puedan convertirse.

### Punto de control

Observar algunos registros es útil, pero no basta para confirmar un formato. Los valores excepcionales suelen aparecer fuera de la muestra y pueden ser precisamente los que impiden una conversión completa.

## La apariencia no garantiza el tipo

Una columna puede verse numérica cuando obsertrabajaremos conlgunas filas, pero eso no garantiza que Pandas pueda tratarla como una columna numérica.

Por ejemplo, `Quantity`, `Price Per Unit` y `Total Spent` representan valores que deberían permitir cálculos. Aun así, como fueron cargadas como `object`, es necesario revisar si pueden usarse directamente en operaciones numéricas.

Intentemos calcular algunas estadísticas sobre `Price Per Unit`.

### Punto de control

Una conversión controlada conserva la evidencia de los casos que no pudieron interpretarse. Es preferible conocer esos casos y revisarlos que ocultarlos mediante una transformación silenciosa.

In [ ]:
df["Price Per Unit"].describe()

Como la columna fue cargada como `object`, `describe()` produce un resumen propio de datos categóricos o textuales, no un resumen numérico.

En lugar de mostrar medidas como media, mediana, mínimo o máximo, Pandas muestra información como cantidad de valores, cantidad de valores únicos, valor más frecuente y frecuencia del valor más frecuente.

Esto nos indica que, para Pandas, `Price Per Unit` todavía no es una variable numérica.

Algo similar ocurre si intentamos ordenar o comparar valores. Cuando una columna está como texto, el orden puede ser alfabético y no necesariamente numérico. Por esa razón, antes de hacer análisis cuantitativo, es necesario convertir estas columnas al tipo adecuado.

### Punto de control

Las copias permiten comparar el estado original con la versión convertida. Esta práctica facilita regresar a la fuente y probar otra estrategia si el resultado no coincide con el objetivo del análisis.

In [ ]:
df["Price Per Unit"].sort_values().head(10)

Esta salida puede resultar engañosa, porque el ordenamiento se realiza sobre textos.

En análisis de datos, no alcanza con que una columna parezca contener números. Necesitamos que Pandas la reconozca como numérica para poder calcular, comparar y ordenar correctamente.

El siguiente paso será convertir estas columnas usando `pd.to_numeric()`.

### Punto de control

Después de transformar una columna, la validación debe revisar tanto el tipo obtenido como los valores faltantes nuevos. Un tipo correcto no garantiza que todos los registros hayan sido interpretados correctamente.

## Transformar texto en números

Para convertir una columna a formato numérico es posible usar `pd.to_numeric()`.

Esta función intenta transformar los valores de una columna en números. Si todos los valores son compatibles con un formato numérico, la conversión suele ser directa.

Probemos primero con la columna `Price Per Unit`.

### Punto de control

Las fechas requieren atención especial porque una cadena puede parecer ordenada y, aun así, no comportarse como una variable temporal. Convertirlas habilita filtros, agrupaciones y comparaciones cronológicas confiables.

In [ ]:
# pd.to_numeric(df["Price Per Unit"])

Es posible que esta instrucción produzca un error.

Ese error aparece porque la columna no contiene solamente números. También contiene valores textuales problemáticos, como `"ERROR"` o `"UNKNOWN"`. Pandas no puede convertir esos textos en números, porque no representan cantidades válidas.

Este resultado es relevante: confirma que el problema no era solamente el tipo `object`, sino la presencia de valores no numéricos dentro de una columna que debería ser numérica.

Para manejar este tipo de situación es posible usar el parámetro `errors`.

### Punto de control

La calidad del resultado depende de los supuestos utilizados durante la conversión. Cuando aparecen `ERROR`, `UNKNOWN` u otros textos, es necesario distinguirlos de los datos válidos antes de continuar.

## Gestionar valores que no convierten

Cuando usamos `pd.to_numeric()`, es posible indicar qué debe hacer Pandas si encuentra un valor que no puede convertir.

Para eso usamos el parámetro `errors`.

Una opción muy útil es `errors="coerce"`.

Con esta opción, Pandas intenta convertir cada valor a número. Si encuentra un valor que no puede convertir, como `"ERROR"` o `"UNKNOWN"`, lo transforma en `NaN`.

Esto no significa que el problema desaparezca. Significa que los valores no convertibles quedan marcados como faltantes, lo cual nos permite detectarlos y tratarlos con las herramientas que ya conocemos.

### Punto de control

El objetivo no es forzar todas las columnas a un tipo único. Cada variable debe conservar el tipo que corresponda a su significado y a las operaciones que se realizarán después.

In [ ]:
precio_convertido = pd.to_numeric(
    df["Price Per Unit"],
    errors="coerce"
)

precio_convertido.head(15)

A continuación la conversión no se detiene con un error.

Los valores numéricos se transforman correctamente, y los valores que no pudieron convertirse pasan a ser `NaN`.

Podemos revisar el tipo de dato resultante:

### Punto de control

El tipo de una columna determina qué operaciones puede realizar Pandas con seguridad. Por eso la conversión debe evaluarse como parte del diagnóstico y no como un cambio meramente visual.

In [ ]:
precio_convertido.dtype

El resultado ya no es `object`. A continuación tenemos una serie numérica.

También es posible revisar cuántos valores faltantes aparecen después de la conversión:

### Punto de control

Observar algunos registros es útil, pero no basta para confirmar un formato. Los valores excepcionales suelen aparecer fuera de la muestra y pueden ser precisamente los que impiden una conversión completa.

In [ ]:
precio_convertido.isna().sum()

Este conteo incluye los valores que ya eran faltantes y también los valores que se convirtieron en `NaN` porque no pudieron interpretarse como números.

Por esa razón, después de usar `errors="coerce"`, siempre debemos verificar cuántos valores quedaron como faltantes. La conversión no debe ocultar el problema: debe hacerlo más fácil de diagnosticar.

### Punto de control

Una conversión controlada conserva la evidencia de los casos que no pudieron interpretarse. Es preferible conocer esos casos y revisarlos que ocultarlos mediante una transformación silenciosa.

## Aplicar la conversión a varias variables

En nuestro dataset no es necesario convertir una sola columna. Hay tres columnas que deberían poder tratarse como numéricas:

```text
Quantity
Price Per Unit
Total Spent
```

Como no queremos modificar el dataset original directamente, trabajaremos con crear una copia llamada `df_convertido`.

Luego aplicaremos `pd.to_numeric()` a cada una de esas columnas.

### Punto de control

Las copias permiten comparar el estado original con la versión convertida. Esta práctica facilita regresar a la fuente y probar otra estrategia si el resultado no coincide con el objetivo del análisis.

In [ ]:
df_convertido = df.copy()

columnas_numericas = [
    "Quantity",
    "Price Per Unit",
    "Total Spent"
]

for columna in columnas_numericas:
    df_convertido[columna] = pd.to_numeric(
        df_convertido[columna],
        errors="coerce"
    )

df_convertido[columnas_numericas].head()

A continuación las columnas seleccionadas fueron convertidas a formato numérico dentro de `df_convertido`.

Usamos un bucle `for` para aplicar la misma transformación a varias columnas. Esto evita repetir tres veces una estructura muy parecida.

La lógica fue la misma en cada caso: intentar convertir los valores a número y transformar en `NaN` aquellos valores que no podían interpretarse como numéricos.

Después de convertir, debemos verificar los tipos de datos.

### Punto de control

Después de transformar una columna, la validación debe revisar tanto el tipo obtenido como los valores faltantes nuevos. Un tipo correcto no garantiza que todos los registros hayan sido interpretados correctamente.

In [ ]:
df_convertido[columnas_numericas].dtypes

A continuación estas columnas ya no aparecen como `object`. Pandas puede tratarlas como columnas numéricas.

Esto nos permite calcular estadísticas, ordenar correctamente, comparar valores y construir nuevas columnas a partir de operaciones matemáticas.

Aun así, todavía debemos revisar algo relevante: al usar `errors="coerce"`, algunos valores problemáticos pueden haberse convertido en `NaN`. Por esa razón, después de convertir, siempre es necesario verificar los faltantes.

### Punto de control

Las fechas requieren atención especial porque una cadena puede parecer ordenada y, aun así, no comportarse como una variable temporal. Convertirlas habilita filtros, agrupaciones y comparaciones cronológicas confiables.

## Revisar el efecto de la conversión

Cuando usamos `pd.to_numeric()` con `errors="coerce"`, los valores que no pueden convertirse pasan a ser `NaN`.

Eso significa que la conversión puede aumentar la cantidad de valores faltantes detectados por Pandas. No porque hayamos perdido datos, sino porque valores como `"ERROR"` o `"UNKNOWN"` fueron transformados en faltantes reconocidos.

Por esa razón, resulta útil comparar los faltantes antes y después de la conversión.

### Punto de control

La calidad del resultado depende de los supuestos utilizados durante la conversión. Cuando aparecen `ERROR`, `UNKNOWN` u otros textos, es necesario distinguirlos de los datos válidos antes de continuar.

In [ ]:
faltantes_antes = df[columnas_numericas].isna().sum()
faltantes_despues = df_convertido[columnas_numericas].isna().sum()

comparacion_faltantes = pd.DataFrame({
    "faltantes_antes": faltantes_antes,
    "faltantes_despues": faltantes_despues,
    "nuevos_faltantes": faltantes_despues - faltantes_antes
})

comparacion_faltantes

La columna `faltantes_antes` muestra los valores que Pandas ya reconocía como faltantes antes de la conversión.

La columna `faltantes_despues` muestra los faltantes luego de convertir las columnas a formato numérico.

La columna `nuevos_faltantes` indica cuántos valores adicionales pasaron a ser `NaN` porque no pudieron convertirse.

Esta comparación es muy relevante. Si solo miramos el resultado final, podríamos pensar que la columna simplemente tenía muchos faltantes. Pero al comparar antes y después vemos que una parte de esos faltantes proviene de valores problemáticos que estaban escritos como texto.

La conversión de tipos no solo cambia el formato de una columna. También puede revelar problemas de calidad que antes estaban ocultos.

### Punto de control

El objetivo no es forzar todas las columnas a un tipo único. Cada variable debe conservar el tipo que corresponda a su significado y a las operaciones que se realizarán después.

## Analizar los datos ya convertidos

Una vez convertidas las columnas numéricas, Pandas puede analizarlas como cantidades reales.

Antes de la conversión, `describe()` sobre `Price Per Unit` producía un resumen de tipo textual. A continuación, sobre `df_convertido`, es posible obtener estadísticas numéricas.

### Punto de control

El tipo de una columna determina qué operaciones puede realizar Pandas con seguridad. Por eso la conversión debe evaluarse como parte del diagnóstico y no como un cambio meramente visual.

In [ ]:
df_convertido["Price Per Unit"].describe()

A continuación la salida incluye medidas como media, desviación estándar, mínimo, cuartiles y máximo.

Esto confirma que la columna ya puede usarse como una variable numérica.

También es posible obtener un resumen de las tres columnas convertidas:

### Punto de control

Observar algunos registros es útil, pero no basta para confirmar un formato. Los valores excepcionales suelen aparecer fuera de la muestra y pueden ser precisamente los que impiden una conversión completa.

In [ ]:
df_convertido[columnas_numericas].describe()

Este resumen permite revisar rangos y valores generales de las columnas numéricas.

A partir de esta conversión, es posible hacer operaciones que antes no eran confiables. Por ejemplo, es posible ordenar por precio unitario de mayor a menor:

### Punto de control

Una conversión controlada conserva la evidencia de los casos que no pudieron interpretarse. Es preferible conocer esos casos y revisarlos que ocultarlos mediante una transformación silenciosa.

In [ ]:
df_convertido.sort_values("Price Per Unit", ascending=False).head()

También es posible crear o verificar relaciones entre columnas numéricas.

Por ejemplo, más adelante podremos revisar si `Total Spent` coincide con `Quantity * Price Per Unit`.

La conversión de tipos no es solo un cambio técnico. Es lo que permite que ciertas columnas puedan participar correctamente en cálculos, comparaciones, ordenamientos y validaciones.

### Punto de control

Las copias permiten comparar el estado original con la versión convertida. Esta práctica facilita regresar a la fuente y probar otra estrategia si el resultado no coincide con el objetivo del análisis.

## Convertir fechas a formato temporal

Además de columnas numéricas, muchos datasets incluyen columnas de fecha.

En nuestro caso, la columna `Transaction Date` representa la fecha de cada transacción. Aun así, al cargar el archivo, Pandas la interpretó como `object`. Eso significa que, por ahora, la está tratando como texto.

Podemos verificarlo:

### Punto de control

Después de transformar una columna, la validación debe revisar tanto el tipo obtenido como los valores faltantes nuevos. Un tipo correcto no garantiza que todos los registros hayan sido interpretados correctamente.

In [ ]:
df_convertido["Transaction Date"].dtype

Aunque los valores se vean como fechas, todavía no son datos temporales para Pandas.

Para convertir una columna a formato de fecha usamos `pd.to_datetime()`.

### Punto de control

Las fechas requieren atención especial porque una cadena puede parecer ordenada y, aun así, no comportarse como una variable temporal. Convertirlas habilita filtros, agrupaciones y comparaciones cronológicas confiables.

In [ ]:
fecha_convertida = pd.to_datetime(
    df_convertido["Transaction Date"],
    errors="coerce"
)

fecha_convertida.head()

La función `pd.to_datetime()` intenta interpretar los valores como fechas.

Usamos nuevamente `errors="coerce"` para evitar que la conversión se detenga si aparece un valor que no puede interpretarse como fecha. En esos casos, Pandas convierte el valor problemático en `NaT`.

`NaT` significa *Not a Time* y es el equivalente temporal de un valor faltante. Así como `NaN` representa un faltante en muchos tipos de columnas, `NaT` representa una fecha faltante o no interpretable.

A continuación es posible revisar el tipo de dato resultante:

### Punto de control

La calidad del resultado depende de los supuestos utilizados durante la conversión. Cuando aparecen `ERROR`, `UNKNOWN` u otros textos, es necesario distinguirlos de los datos válidos antes de continuar.

In [ ]:
fecha_convertida.dtype

El tipo `datetime64[ns]` indica que Pandas ya interpreta la columna como una fecha.

Podemos guardar esta conversión en el `DataFrame` convertido:

### Punto de control

El objetivo no es forzar todas las columnas a un tipo único. Cada variable debe conservar el tipo que corresponda a su significado y a las operaciones que se realizarán después.

In [ ]:
df_convertido["Transaction Date"] = pd.to_datetime(
    df_convertido["Transaction Date"],
    errors="coerce"
)

df_convertido["Transaction Date"].dtype

A partir de esta conversión, la columna `Transaction Date` puede usarse como dato temporal.

Todavía no trabajaremos con profundizar en operaciones con fechas. Más adelante podremos extraer el año, el mes, el día, ordenar correctamente las transacciones en el tiempo o agrupar ventas por período.

En este capítulo nos alcanza con comprender la idea principal: una fecha escrita como texto no es lo mismo que una fecha interpretada como dato temporal por Pandas.

### Punto de control

El tipo de una columna determina qué operaciones puede realizar Pandas con seguridad. Por eso la conversión debe evaluarse como parte del diagnóstico y no como un cambio meramente visual.

## Comprobar las fechas interpretadas

Así como hicimos con las columnas numéricas, después de convertir una columna de fecha resulta útil verificar el resultado.

Primero es posible revisar cuántos valores faltantes tenía originalmente la columna `Transaction Date` y cuántos valores faltantes o no interpretables aparecen después de la conversión.

### Punto de control

Observar algunos registros es útil, pero no basta para confirmar un formato. Los valores excepcionales suelen aparecer fuera de la muestra y pueden ser precisamente los que impiden una conversión completa.

In [ ]:
faltantes_fecha_antes = df["Transaction Date"].isna().sum()
faltantes_fecha_despues = df_convertido["Transaction Date"].isna().sum()

print("Faltantes antes de convertir:")
print(faltantes_fecha_antes)

print()

print("Faltantes después de convertir:")
print(faltantes_fecha_despues)

Si la cantidad de faltantes aumenta después de la conversión, eso significa que algunos valores que antes estaban escritos como texto no pudieron interpretarse como fechas y fueron convertidos en `NaT`.

También es posible revisar algunas estadísticas básicas de la columna convertida.

### Punto de control

Una conversión controlada conserva la evidencia de los casos que no pudieron interpretarse. Es preferible conocer esos casos y revisarlos que ocultarlos mediante una transformación silenciosa.

In [ ]:
df_convertido["Transaction Date"].describe()

El resumen de una columna temporal nos permite observar, entre otras cosas, la fecha mínima y la fecha máxima registradas.

Esto puede ser útil para detectar fechas fuera de rango o valores extraños. Por ejemplo, si un dataset de ventas de 2024 tuviera una fecha del año 2099, esa fecha debería llamar nuestra atención.

También es posible ordenar el dataset por fecha:

### Punto de control

Las copias permiten comparar el estado original con la versión convertida. Esta práctica facilita regresar a la fuente y probar otra estrategia si el resultado no coincide con el objetivo del análisis.

In [ ]:
df_convertido.sort_values("Transaction Date").head()

A continuación el ordenamiento se realiza con una columna temporal, no con texto.

En muchos casos esto no cambia visualmente el resultado si las fechas estaban escritas en un formato ordenable, como `YYYY-MM-DD`. Pero conceptualmente sí es relevante: Pandas ahora reconoce la columna como fecha y puede aplicar operaciones temporales sobre ella.

La conversión de fechas será la base para trabajar más adelante con análisis por día, mes, año o períodos.

### Punto de control

Después de transformar una columna, la validación debe revisar tanto el tipo obtenido como los valores faltantes nuevos. Un tipo correcto no garantiza que todos los registros hayan sido interpretados correctamente.

## Confirmar la estructura final

Después de convertir columnas numéricas y temporales, resulta útil revisar nuevamente la estructura general del `DataFrame`.

El objetivo es comprobar que las columnas que necesitaban conversión ya tengan tipos de datos más adecuados para el análisis.

### Punto de control

Las fechas requieren atención especial porque una cadena puede parecer ordenada y, aun así, no comportarse como una variable temporal. Convertirlas habilita filtros, agrupaciones y comparaciones cronológicas confiables.

In [ ]:
df_convertido.dtypes

A continuación deberíamos observar que:

```text
Quantity           → tipo numérico
Price Per Unit     → tipo numérico
Total Spent        → tipo numérico
Transaction Date   → tipo fecha
```

Mientras tanto, columnas como `Transaction ID`, `Item`, `Payment Method` y `Location` pueden seguir como `object`, porque representan identificadores o categorías textuales.

También es posible usar `info()` para ver la estructura completa:

### Punto de control

La calidad del resultado depende de los supuestos utilizados durante la conversión. Cuando aparecen `ERROR`, `UNKNOWN` u otros textos, es necesario distinguirlos de los datos válidos antes de continuar.

In [ ]:
df_convertido.info()

Esta revisión final permite confirmar que el dataset está en una mejor condición para el análisis.

Convertir tipos no significa que el dataset ya esté completamente limpio. Todavía pueden quedar valores faltantes, categorías problemáticas, duplicados u otras inconsistencias. Pero sí significa que algunas columnas ya están en un formato más apropiado para operar con ellas.

A partir de este momento, las columnas numéricas pueden participar en cálculos y comparaciones, y la columna de fecha puede usarse para análisis temporal.

La validación final es una parte esencial del proceso. No basta con aplicar conversiones: es necesario comprobar que el resultado tenga sentido.

### Punto de control

El objetivo no es forzar todas las columnas a un tipo único. Cada variable debe conservar el tipo que corresponda a su significado y a las operaciones que se realizarán después.

## Errores que resulta útil evitar

Al convertir tipos de datos, uno de los errores más comunes es asumir que una columna está bien solo porque sus valores “parecen” numéricos o “parecen” fechas.

Para Pandas, no alcanza con la apariencia visual. Si una columna fue cargada como `object`, puede contener textos, valores problemáticos o datos mezclados. Antes de usarla en cálculos, resulta útil revisar su tipo y convertirla si corresponde.

Otro error frecuente es usar `pd.to_numeric()` o `pd.to_datetime()` sin verificar qué ocurrió después. Cuando usamos `errors="coerce"`, los valores que no pueden convertirse pasan a ser `NaN` o `NaT`. Eso es útil porque evita que la conversión se detenga, pero también puede aumentar la cantidad de valores faltantes.

Por esa razón, después de convertir, siempre resulta útil comparar faltantes antes y después.

También debemos evitar sobrescribir el dataset original demasiado pronto. En este capítulo trabajamos con una copia llamada `df_convertido`, lo que nos permitió transformar columnas sin perder el punto de partida.

Otro error posible es convertir columnas que no necesitan conversión. Por ejemplo, `Transaction ID` contiene identificadores. Aunque tenga números dentro del texto, no resulta útil tratarlo como una variable numérica si no representa una cantidad. Un identificador no se suma, no se promedia y no se interpreta como magnitud.

Algo similar ocurre con ciertas categorías codificadas. Que una columna tenga números no significa necesariamente que sea numérica en sentido analítico. Siempre debemos preguntarnos qué representa la columna.

Una rutina razonable para convertir tipos podría ser:

```text
revisar tipos actuales
identificar columnas que necesitan conversión
observar valores problemáticos
convertir en una copia del DataFrame
verificar nuevos tipos
comparar faltantes antes y después
confirmar que las columnas convertidas se comportan como esperamos
```

La conversión de tipos es una parte central de la preparación de datos. Si las columnas no tienen el tipo adecuado, muchas operaciones posteriores pueden fallar o producir resultados engañosos.

### Punto de control

El tipo de una columna determina qué operaciones puede realizar Pandas con seguridad. Por eso la conversión debe evaluarse como parte del diagnóstico y no como un cambio meramente visual.

## Síntesis del proceso

En esta práctica trabajamos con la conversión de tipos de datos.

Partimos de una idea relevante: no alcanza con que una columna parezca numérica o parezca una fecha. Para que Pandas pueda operar correctamente con esos valores, necesita interpretarlos con el tipo de dato adecuado.

Primero revisamos los tipos de datos con:

```python
df.dtypes
```

y con:

```python id="sh4lzh"
df.info()
```

La salida de `info()` mostró que todas las columnas del dataset habían sido cargadas como `object`. Esto era esperable para columnas como `Transaction ID`, `Item`, `Payment Method` o `Location`, pero no para columnas como `Quantity`, `Price Per Unit`, `Total Spent` o `Transaction Date`.

Después observamos las columnas que deberían ser numéricas:

```python id="1nfyih"
df[["Quantity", "Price Per Unit", "Total Spent"]].head(15)
```

y vimos que podían contener valores problemáticos como `"ERROR"` o `"UNKNOWN"`. Esa mezcla de números y textos explica por qué Pandas las cargó como `object`.

También comprobamos que una columna cargada como texto no produce un resumen numérico adecuado. Por ejemplo, `describe()` sobre `Price Per Unit` antes de la conversión genera un resumen propio de datos textuales, no estadísticas numéricas.

Luego usamos `pd.to_numeric()` para convertir valores a formato numérico. Como algunas celdas contenían valores no convertibles, usamos:

```python id="47j5p8"
pd.to_numeric(df["Price Per Unit"], errors="coerce")
```

El parámetro `errors="coerce"` permite transformar en `NaN` los valores que no pueden convertirse a número.

Más adelante aplicamos esa conversión a varias columnas:

```python id="vwgf5t"
columnas_numericas = [
    "Quantity",
    "Price Per Unit",
    "Total Spent"
]

for columna in columnas_numericas:
    df_convertido[columna] = pd.to_numeric(
        df_convertido[columna],
        errors="coerce"
    )
```

Después verificamos los tipos resultantes y comparamos los faltantes antes y después de la conversión. Esta comparación fue relevante porque algunos valores problemáticos que antes estaban escritos como texto pasaron a ser `NaN`.

También convertimos la columna `Transaction Date` usando:

```python id="cu2qm3"
pd.to_datetime(df_convertido["Transaction Date"], errors="coerce")
```

Para este ejemplo, los valores que no pudieron interpretarse como fechas pasaron a ser `NaT`, que representa valores temporales faltantes o no interpretables.

Finalmente validamos la estructura final con `dtypes` e `info()`, confirmando que las columnas numéricas y la columna de fecha ya estaban en tipos más adecuados para el análisis.

La idea principal de este capítulo fue:

```text id="wl8mah"
Convertir tipos de datos permite que Pandas interprete correctamente qué clase de información contiene cada columna.
```

Este cambio de tipo es necesaria para calcular, ordenar, comparar, validar y analizar datos de manera confiable.

### Punto de control

Observar algunos registros es útil, pero no basta para confirmar un formato. Los valores excepcionales suelen aparecer fuera de la muestra y pueden ser precisamente los que impiden una conversión completa.

## Siguiente etapa

Ya trabajamos con valores faltantes, duplicados, limpieza de textos y conversión de tipos de datos.

El siguiente paso será profundizar en fechas y datos temporales. A continuación que sabemos convertir una columna a formato de fecha, es posible aprender a extraer información útil de ella: año, mes, día, día de la semana o períodos.

También trabajaremos con ver por qué las fechas son especialmente relevantes en muchos análisis, ya que permiten estudiar evolución, estacionalidad, cambios en el tiempo y comportamiento por períodos.

### Punto de control

Una conversión controlada conserva la evidencia de los casos que no pudieron interpretarse. Es preferible conocer esos casos y revisarlos que ocultarlos mediante una transformación silenciosa.